In [1]:
import os, time, requests
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4"
)

cities = {
    "Amsterdam": {"lat": 52.3676, "lon": 4.9041,  "country": "NLD"},
    "Barcelona": {"lat": 41.3874, "lon": 2.1686,  "country": "ESP"},
    "Berlin":    {"lat": 52.5200, "lon": 13.4050, "country": "DEU"},
    "Columbus":  {"lat": 39.9612, "lon": -82.9988, "country": "USA"},
}

In [2]:
indicators = {
    "NY.GDP.PCAP.CD": "gdp_per_capita_usd",
    "ST.INT.ARVL":    "international_tourism_arrivals",
    "FP.CPI.TOTL.ZG": "inflation_pct",
}
countries = ";".join(c["country"] for c in cities.values())

rows = []
for code, name in indicators.items():
    url = f"https://api.worldbank.org/v2/country/{countries}/indicator/{code}"
    r = requests.get(url, params={"format": "json", "per_page": 500, "date": "2018:2025"}, timeout=60)
    r.raise_for_status()
    for item in r.json()[1]:
        rows.append({
            "country_code": item["countryiso3code"],
            "country": item["country"]["value"],
            "year": item["date"],
            "indicator": name,
            "value": item["value"],
        })

wb = pd.DataFrame(rows)
print(wb.pivot_table(index=["country", "year"], columns="indicator", values="value").tail(12))

indicator           gdp_per_capita_usd  inflation_pct  \
country       year                                      
Spain         2022        30319.498720       8.390419   
              2023        33493.222541       3.532473   
              2024        35326.768307       2.774070   
              2025        38627.247215       2.673267   
United States 2018        62875.666138       2.442583   
              2019        65227.956591       1.812210   
              2020        64465.297142       1.233584   
              2021        71441.231981       4.697859   
              2022        78008.689581       8.002800   
              2023        82586.784771       4.116338   
              2024        86169.664158       2.949525   
              2025        90026.516301            NaN   

indicator           international_tourism_arrivals  
country       year                                  
Spain         2022                             NaN  
              2023                        

In [3]:
wb.to_sql("raw_economic_indicators", engine, if_exists="replace", index=False)
with engine.connect() as conn:
    print(conn.execute(text("SELECT country, COUNT(*) FROM raw_economic_indicators GROUP BY country")).fetchall())

[('Germany', 24), ('Spain', 24), ('Netherlands', 24), ('United States', 24)]
